In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings("ignore")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.grid"] = True

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
df = pd.read_csv("DailyDelhiClimate.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Missing values:")
print(df.isna().sum())
df.head()

In [ ]:
display(df.describe().T)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(df["date"], df["meantemp"], linewidth=1.5)
axes[0, 0].set_title("Mean Temperature")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Temperature")

axes[0, 1].plot(df["date"], df["humidity"], linewidth=1.5)
axes[0, 1].set_title("Humidity")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Humidity")

axes[1, 0].plot(df["date"], df["wind_speed"], linewidth=1.5)
axes[1, 0].set_title("Wind Speed")
axes[1, 0].set_xlabel("Date")
axes[1, 0].set_ylabel("Wind Speed")

axes[1, 1].plot(df["date"], df["meanpressure"], linewidth=1.5)
axes[1, 1].set_title("Mean Pressure")
axes[1, 1].set_xlabel("Date")
axes[1, 1].set_ylabel("Pressure")

plt.tight_layout()
plt.show()

In [ ]:
work = df.copy()
work["dayofyear"] = work["date"].dt.dayofyear
work["sin_day"] = np.sin(2 * np.pi * work["dayofyear"] / 365.25)
work["cos_day"] = np.cos(2 * np.pi * work["dayofyear"] / 365.25)

features = ["humidity", "wind_speed", "meanpressure", "sin_day", "cos_day"]
target = "meantemp"

X_all = work[features].to_numpy(dtype=np.float32)
y_all = work[target].to_numpy(dtype=np.float32)

split_index = int(len(work) * 0.8)

X_train = X_all[:split_index]
X_test = X_all[split_index:]
y_train = y_all[:split_index]
y_test = y_all[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Training samples:", len(X_train_scaled))
print("Testing samples:", len(X_test_scaled))
print("Input features:", features)

In [ ]:
def loss_surface(w1, w2):
    return 0.5 * (w1**2 + 25.0 * w2**2) + 0.15 * np.sin(1.2 * w1) * np.sin(1.0 * w2)

def loss_value(point):
    w1, w2 = point
    return 0.5 * (w1**2 + 25.0 * w2**2) + 0.15 * np.sin(1.2 * w1) * np.sin(1.0 * w2)

def gradient_value(point):
    w1, w2 = point
    dw1 = w1 + 0.18 * np.cos(1.2 * w1) * np.sin(w2)
    dw2 = 25.0 * w2 + 0.15 * np.sin(1.2 * w1) * np.cos(w2)
    return np.array([dw1, dw2], dtype=float)

x = np.linspace(-6, 6, 260)
y = np.linspace(-2.4, 2.4, 220)
Xg, Yg = np.meshgrid(x, y)
Zg = loss_surface(Xg, Yg)

minimum_index = np.unravel_index(np.argmin(Zg), Zg.shape)
minimum_point = np.array([Xg[minimum_index], Yg[minimum_index]])

print("Approximate visual minimum:", minimum_point)
print("Minimum loss:", Zg[minimum_index])

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(Xg, Yg, Zg, cmap="turbo", alpha=0.75, linewidth=0)
ax.scatter(minimum_point[0], minimum_point[1], loss_value(minimum_point), s=140, marker="*", color="red")
ax.set_title("3D Ill-Conditioned Loss Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
fig.colorbar(surface, ax=ax, shrink=0.6, pad=0.1, label="Loss")
ax.view_init(elev=37, azim=-55)
plt.show()

fig, ax = plt.subplots(figsize=(13, 9))
levels = np.linspace(np.percentile(Zg, 2), np.percentile(Zg, 96), 45)
cf = ax.contourf(Xg, Yg, Zg, levels=levels, cmap="turbo")
ax.contour(Xg, Yg, Zg, levels=levels[::3], colors="white", linewidths=0.55, alpha=0.35)
ax.scatter(minimum_point[0], minimum_point[1], s=130, marker="*", color="red")
ax.set_title("Contour Map of the Same Loss Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_aspect("equal")
plt.colorbar(cf, ax=ax, label="Loss")
plt.show()

In [ ]:
def optimize_sgd(start, lr=0.025, steps=80):
    w = np.array(start, dtype=float)
    path = [w.copy()]
    losses = [loss_value(w)]
    for _ in range(steps):
        g = gradient_value(w)
        w = w - lr * g
        path.append(w.copy())
        losses.append(loss_value(w))
    return np.array(path), np.array(losses)

def optimize_momentum(start, lr=0.012, beta=0.90, steps=80):
    w = np.array(start, dtype=float)
    v = np.zeros(2, dtype=float)
    path = [w.copy()]
    losses = [loss_value(w)]
    velocities = [v.copy()]
    for _ in range(steps):
        g = gradient_value(w)
        v = beta * v + g
        w = w - lr * v
        path.append(w.copy())
        losses.append(loss_value(w))
        velocities.append(v.copy())
    return np.array(path), np.array(losses), np.array(velocities)

def optimize_nag(start, lr=0.012, beta=0.90, steps=80):
    w = np.array(start, dtype=float)
    v = np.zeros(2, dtype=float)
    path = [w.copy()]
    losses = [loss_value(w)]
    look_ahead = [w.copy()]
    velocities = [v.copy()]
    for _ in range(steps):
        look = w - lr * beta * v
        g = gradient_value(look)
        v = beta * v + g
        w = w - lr * v
        path.append(w.copy())
        losses.append(loss_value(w))
        look_ahead.append(look.copy())
        velocities.append(v.copy())
    return np.array(path), np.array(losses), np.array(look_ahead), np.array(velocities)

start = np.array([-5.0, 2.0])

sgd_path, sgd_losses = optimize_sgd(start)
momentum_path, momentum_losses, momentum_velocity = optimize_momentum(start)
nag_path, nag_losses, nag_look, nag_velocity = optimize_nag(start)

print("SGD final loss:", sgd_losses[-1])
print("Momentum final loss:", momentum_losses[-1])
print("NAG final loss:", nag_losses[-1])

In [ ]:
summary_surface = pd.DataFrame({
    "Optimizer": ["SGD", "Momentum", "NAG"],
    "Initial Loss": [sgd_losses[0], momentum_losses[0], nag_losses[0]],
    "Final Loss": [sgd_losses[-1], momentum_losses[-1], nag_losses[-1]],
    "Best Loss": [sgd_losses.min(), momentum_losses.min(), nag_losses.min()],
    "Final w1": [sgd_path[-1, 0], momentum_path[-1, 0], nag_path[-1, 0]],
    "Final w2": [sgd_path[-1, 1], momentum_path[-1, 1], nag_path[-1, 1]],
})
summary_surface

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
levels = np.linspace(np.percentile(Zg, 2), np.percentile(Zg, 96), 45)
cf = ax.contourf(Xg, Yg, Zg, levels=levels, cmap="turbo")
ax.contour(Xg, Yg, Zg, levels=levels[::3], colors="white", linewidths=0.5, alpha=0.28)

ax.plot(sgd_path[:, 0], sgd_path[:, 1], linewidth=3, label="SGD")
ax.plot(momentum_path[:, 0], momentum_path[:, 1], linewidth=3, label="Momentum")
ax.plot(nag_path[:, 0], nag_path[:, 1], linewidth=3, label="NAG")

ax.scatter(*start, s=150, marker="*", color="black", label="Start", zorder=6)
ax.scatter(*minimum_point, s=150, marker="X", color="red", label="Minimum", zorder=6)

for path, label in [(sgd_path, "SGD"), (momentum_path, "Momentum"), (nag_path, "NAG")]:
    ax.scatter(path[-1, 0], path[-1, 1], s=110, marker="o", label=f"{label} End")

ax.set_title("Optimizer Travel Paths on the Contour Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_aspect("equal")
ax.legend()
plt.colorbar(cf, ax=ax, label="Loss")
plt.show()


In [ ]:
stride = 4
Xs = Xg[::stride, ::stride]
Ys = Yg[::stride, ::stride]
Zs = Zg[::stride, ::stride]

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(Xs, Ys, Zs, cmap="turbo", alpha=0.68, linewidth=0)
ax.plot(sgd_path[:, 0], sgd_path[:, 1], [loss_value(p) for p in sgd_path], linewidth=3.2, label="SGD")
ax.plot(momentum_path[:, 0], momentum_path[:, 1], [loss_value(p) for p in momentum_path], linewidth=3.2, label="Momentum")
ax.plot(nag_path[:, 0], nag_path[:, 1], [loss_value(p) for p in nag_path], linewidth=3.2, label="NAG")
ax.scatter(*start, loss_value(start), s=130, marker="*", color="black")
ax.scatter(*minimum_point, loss_value(minimum_point), s=150, marker="X", color="red")

ax.set_title("3D Optimization Travel: SGD vs Momentum vs NAG")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
fig.colorbar(surface, ax=ax, shrink=0.55, pad=0.08, label="Loss")
ax.legend()
ax.view_init(elev=38, azim=-60)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
levels = np.linspace(np.percentile(Zg, 2), np.percentile(Zg, 96), 45)
cf = ax.contourf(Xg, Yg, Zg, levels=levels, cmap="turbo")
ax.contour(Xg, Yg, Zg, levels=levels[::3], colors="white", linewidths=0.5, alpha=0.28)

line_sgd, = ax.plot([], [], linewidth=3, label="SGD")
line_mom, = ax.plot([], [], linewidth=3, label="Momentum")
line_nag, = ax.plot([], [], linewidth=3, label="NAG")
point_sgd, = ax.plot([], [], marker="o", markersize=9)
point_mom, = ax.plot([], [], marker="o", markersize=9)
point_nag, = ax.plot([], [], marker="o", markersize=9)

ax.scatter(*start, s=140, marker="*", color="black", zorder=5, label="Start")
ax.scatter(*minimum_point, s=140, marker="X", color="red", zorder=5, label="Minimum")
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_title("Animated Optimizer Travel on Contours")
ax.set_aspect("equal")
ax.legend()
plt.colorbar(cf, ax=ax, label="Loss")

def init_animation():
    for artist in [line_sgd, line_mom, line_nag, point_sgd, point_mom, point_nag]:
        artist.set_data([], [])
    return line_sgd, line_mom, line_nag, point_sgd, point_mom, point_nag

def update_animation(frame):
    n = frame + 1
    line_sgd.set_data(sgd_path[:n, 0], sgd_path[:n, 1])
    line_mom.set_data(momentum_path[:n, 0], momentum_path[:n, 1])
    line_nag.set_data(nag_path[:n, 0], nag_path[:n, 1])
    point_sgd.set_data([sgd_path[n-1, 0]], [sgd_path[n-1, 1]])
    point_mom.set_data([momentum_path[n-1, 0]], [momentum_path[n-1, 1]])
    point_nag.set_data([nag_path[n-1, 0]], [nag_path[n-1, 1]])
    ax.set_title(f"Animated Optimizer Travel — Step {frame}")
    return line_sgd, line_mom, line_nag, point_sgd, point_mom, point_nag

anim = FuncAnimation(
    fig,
    update_animation,
    init_func=init_animation,
    frames=len(sgd_path),
    interval=90,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(Xs, Ys, Zs, cmap="turbo", alpha=0.62, linewidth=0)

line_sgd3d, = ax.plot([], [], [], linewidth=3.3, label="SGD")
line_mom3d, = ax.plot([], [], [], linewidth=3.3, label="Momentum")
line_nag3d, = ax.plot([], [], [], linewidth=3.3, label="NAG")
point_sgd3d, = ax.plot([], [], [], marker="o", markersize=8)
point_mom3d, = ax.plot([], [], [], marker="o", markersize=8)
point_nag3d, = ax.plot([], [], [], marker="o", markersize=8)

ax.scatter(start[0], start[1], loss_value(start), s=140, marker="*", color="black")
ax.scatter(minimum_point[0], minimum_point[1], loss_value(minimum_point), s=150, marker="X", color="red")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
ax.set_title("3D Animated Descent: SGD vs Momentum vs NAG")
fig.colorbar(surface, ax=ax, shrink=0.55, pad=0.08, label="Loss")
ax.legend()
ax.view_init(elev=36, azim=-60)

sgd_z = np.array([loss_value(p) for p in sgd_path])
mom_z = np.array([loss_value(p) for p in momentum_path])
nag_z = np.array([loss_value(p) for p in nag_path])

def init_3d():
    for artist in [line_sgd3d, line_mom3d, line_nag3d, point_sgd3d, point_mom3d, point_nag3d]:
        artist.set_data([], [])
        artist.set_3d_properties([])
    return line_sgd3d, line_mom3d, line_nag3d, point_sgd3d, point_mom3d, point_nag3d

def update_3d(frame):
    n = frame + 1

    line_sgd3d.set_data(sgd_path[:n, 0], sgd_path[:n, 1])
    line_sgd3d.set_3d_properties(sgd_z[:n])
    point_sgd3d.set_data([sgd_path[n-1, 0]], [sgd_path[n-1, 1]])
    point_sgd3d.set_3d_properties([sgd_z[n-1]])

    line_mom3d.set_data(momentum_path[:n, 0], momentum_path[:n, 1])
    line_mom3d.set_3d_properties(mom_z[:n])
    point_mom3d.set_data([momentum_path[n-1, 0]], [momentum_path[n-1, 1]])
    point_mom3d.set_3d_properties([mom_z[n-1]])

    line_nag3d.set_data(nag_path[:n, 0], nag_path[:n, 1])
    line_nag3d.set_3d_properties(nag_z[:n])
    point_nag3d.set_data([nag_path[n-1, 0]], [nag_path[n-1, 1]])
    point_nag3d.set_3d_properties([nag_z[n-1]])

    ax.set_title(f"3D Animated Descent — Step {frame}")
    return line_sgd3d, line_mom3d, line_nag3d, point_sgd3d, point_mom3d, point_nag3d

anim3d = FuncAnimation(
    fig,
    update_3d,
    init_func=init_3d,
    frames=len(sgd_path),
    interval=90,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(anim3d.to_jshtml())

In [ ]:
def build_ann():
    return keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1)
    ])

tf.random.set_seed(123)
base_model = build_ann()
initial_weights = base_model.get_weights()

sgd_model = build_ann()
momentum_model = build_ann()
nag_model = build_ann()

sgd_model.set_weights(initial_weights)
momentum_model.set_weights(initial_weights)
nag_model.set_weights(initial_weights)

sgd_optimizer = keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.0,
    nesterov=False,
    clipnorm=1.0
)

momentum_optimizer = keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=False,
    clipnorm=1.0
)

nag_optimizer = keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=True,
    clipnorm=1.0
)

sgd_model.compile(optimizer=sgd_optimizer, loss="mse", metrics=["mae"])
momentum_model.compile(optimizer=momentum_optimizer, loss="mse", metrics=["mae"])
nag_model.compile(optimizer=nag_optimizer, loss="mse", metrics=["mae"])

print("SGD:", sgd_optimizer.get_config())
print("Momentum:", momentum_optimizer.get_config())
print("NAG:", nag_optimizer.get_config())

In [ ]:
class WeightPathRecorder(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.weights = []
        self.losses = []
        self.val_losses = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        flat = np.concatenate([w.numpy().ravel() for w in self.model.trainable_weights])
        self.weights.append(flat.copy())
        self.losses.append(float(logs.get("loss", np.nan)))
        self.val_losses.append(float(logs.get("val_loss", np.nan)))

sgd_recorder = WeightPathRecorder()
momentum_recorder = WeightPathRecorder()
nag_recorder = WeightPathRecorder()

sgd_history = sgd_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    shuffle=False,
    verbose=0,
    callbacks=[sgd_recorder]
)

momentum_history = momentum_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    shuffle=False,
    verbose=0,
    callbacks=[momentum_recorder]
)

nag_history = nag_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    shuffle=False,
    verbose=0,
    callbacks=[nag_recorder]
)

print("Training complete")

In [ ]:
ann_loss = pd.DataFrame({
    "Epoch": np.arange(1, 101),
    "SGD": sgd_history.history["loss"],
    "Momentum": momentum_history.history["loss"],
    "NAG": nag_history.history["loss"]
})

ann_val_loss = pd.DataFrame({
    "Epoch": np.arange(1, 101),
    "SGD": sgd_history.history["val_loss"],
    "Momentum": momentum_history.history["val_loss"],
    "NAG": nag_history.history["val_loss"]
})

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(ann_loss["Epoch"], ann_loss["SGD"], linewidth=2.5, label="SGD")
axes[0].plot(ann_loss["Epoch"], ann_loss["Momentum"], linewidth=2.5, label="Momentum")
axes[0].plot(ann_loss["Epoch"], ann_loss["NAG"], linewidth=2.5, label="NAG")
axes[0].set_title("ANN Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(ann_val_loss["Epoch"], ann_val_loss["SGD"], linewidth=2.5, label="SGD")
axes[1].plot(ann_val_loss["Epoch"], ann_val_loss["Momentum"], linewidth=2.5, label="Momentum")
axes[1].plot(ann_val_loss["Epoch"], ann_val_loss["NAG"], linewidth=2.5, label="NAG")
axes[1].set_title("ANN Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
pred_sgd = sgd_model.predict(X_test_scaled, verbose=0).ravel()
pred_momentum = momentum_model.predict(X_test_scaled, verbose=0).ravel()
pred_nag = nag_model.predict(X_test_scaled, verbose=0).ravel()

metrics = pd.DataFrame({
    "Optimizer": ["SGD", "Momentum", "NAG"],
    "MAE": [
        mean_absolute_error(y_test, pred_sgd),
        mean_absolute_error(y_test, pred_momentum),
        mean_absolute_error(y_test, pred_nag)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, pred_sgd)),
        np.sqrt(mean_squared_error(y_test, pred_momentum)),
        np.sqrt(mean_squared_error(y_test, pred_nag))
    ],
    "R2": [
        r2_score(y_test, pred_sgd),
        r2_score(y_test, pred_momentum),
        r2_score(y_test, pred_nag)
    ],
    "Final Train MSE": [
        sgd_history.history["loss"][-1],
        momentum_history.history["loss"][-1],
        nag_history.history["loss"][-1]
    ],
    "Final Val MSE": [
        sgd_history.history["val_loss"][-1],
        momentum_history.history["val_loss"][-1],
        nag_history.history["val_loss"][-1]
    ]
})

metrics.round(4)

In [ ]:
from sklearn.decomposition import PCA

all_weights = np.vstack([
    np.vstack(sgd_recorder.weights),
    np.vstack(momentum_recorder.weights),
    np.vstack(nag_recorder.weights)
])

pca = PCA(n_components=2)
all_pc = pca.fit_transform(all_weights)

n_epochs = len(sgd_recorder.weights)

sgd_pc = all_pc[:n_epochs]
momentum_pc = all_pc[n_epochs:2*n_epochs]
nag_pc = all_pc[2*n_epochs:]

sgd_loss_arr = np.array(sgd_recorder.losses)
momentum_loss_arr = np.array(momentum_recorder.losses)
nag_loss_arr = np.array(nag_recorder.losses)

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

ax.plot(sgd_pc[:, 0], sgd_pc[:, 1], sgd_loss_arr, linewidth=3, label="SGD")
ax.plot(momentum_pc[:, 0], momentum_pc[:, 1], momentum_loss_arr, linewidth=3, label="Momentum")
ax.plot(nag_pc[:, 0], nag_pc[:, 1], nag_loss_arr, linewidth=3, label="NAG")

ax.scatter(sgd_pc[0, 0], sgd_pc[0, 1], sgd_loss_arr[0], s=100, marker="*")
ax.set_title("3D ANN Weight-Space Training Trajectories")
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_zlabel("Training MSE")
ax.legend()
ax.view_init(elev=30, azim=-60)
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(111, projection="3d")

ax.plot(sgd_pc[:, 0], sgd_pc[:, 1], sgd_loss_arr, linewidth=3, label="SGD")
ax.plot(momentum_pc[:, 0], momentum_pc[:, 1], momentum_loss_arr, linewidth=3, label="Momentum")
ax.plot(nag_pc[:, 0], nag_pc[:, 1], nag_loss_arr, linewidth=3, label="NAG")

point_sgd_ann, = ax.plot([], [], [], marker="o", markersize=8)
point_mom_ann, = ax.plot([], [], [], marker="o", markersize=8)
point_nag_ann, = ax.plot([], [], [], marker="o", markersize=8)

ax.set_xlim(all_pc[:, 0].min(), all_pc[:, 0].max())
ax.set_ylim(all_pc[:, 1].min(), all_pc[:, 1].max())
ax.set_zlim(
    min(sgd_loss_arr.min(), momentum_loss_arr.min(), nag_loss_arr.min()),
    max(sgd_loss_arr.max(), momentum_loss_arr.max(), nag_loss_arr.max())
)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_zlabel("Training MSE")
ax.set_title("Animated 3D ANN Optimization Trajectory")
ax.view_init(elev=30, azim=-60)
ax.legend()

def init_ann_3d():
    for artist in [point_sgd_ann, point_mom_ann, point_nag_ann]:
        artist.set_data([], [])
        artist.set_3d_properties([])
    return point_sgd_ann, point_mom_ann, point_nag_ann

def update_ann_3d(frame):
    i = frame
    point_sgd_ann.set_data([sgd_pc[i, 0]], [sgd_pc[i, 1]])
    point_sgd_ann.set_3d_properties([sgd_loss_arr[i]])

    point_mom_ann.set_data([momentum_pc[i, 0]], [momentum_pc[i, 1]])
    point_mom_ann.set_3d_properties([momentum_loss_arr[i]])

    point_nag_ann.set_data([nag_pc[i, 0]], [nag_pc[i, 1]])
    point_nag_ann.set_3d_properties([nag_loss_arr[i]])

    ax.set_title(f"Animated ANN Optimization Trajectory — Epoch {i + 1}")
    return point_sgd_ann, point_mom_ann, point_nag_ann

ann_anim = FuncAnimation(
    fig,
    update_ann_3d,
    init_func=init_ann_3d,
    frames=n_epochs,
    interval=100,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(ann_anim.to_jshtml())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(np.arange(1, n_epochs + 1), sgd_loss_arr, linewidth=2.6, label="SGD")
ax.plot(np.arange(1, n_epochs + 1), momentum_loss_arr, linewidth=2.6, label="Momentum")
ax.plot(np.arange(1, n_epochs + 1), nag_loss_arr, linewidth=2.6, label="NAG")
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE")
ax.set_title("Optimizer Convergence Speed on the Same ANN")
ax.legend()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Quantity": [
        "Uses current gradient",
        "Stores velocity",
        "Look-ahead gradient",
        "Can overshoot",
        "Typical purpose"
    ],
    "SGD": [
        "Yes",
        "No",
        "No",
        "Lower",
        "Simple baseline"
    ],
    "Momentum": [
        "Yes",
        "Yes",
        "No",
        "Higher",
        "Accelerate consistent movement"
    ],
    "NAG": [
        "Look-ahead",
        "Yes",
        "Yes",
        "Reduced relative to Momentum",
        "Faster and more controlled momentum updates"
    ]
})

comparison

In [ ]:
best_rmse = metrics.loc[metrics["RMSE"].idxmin(), "Optimizer"]
best_mae = metrics.loc[metrics["MAE"].idxmin(), "Optimizer"]
best_r2 = metrics.loc[metrics["R2"].idxmax(), "Optimizer"]

print("Best RMSE optimizer:", best_rmse)
print("Best MAE optimizer:", best_mae)
print("Best R² optimizer:", best_r2)
print()
print(metrics.round(4).to_string(index=False))